# COOL: an interior open boundary demo

The topography in `cool_topo.nc` spells **COOL** in 20 m deep channels. No letter
touches the domain edge, so every letter starts closed.

This notebook opens two **interior** OBC segments on the **L**. Salty water (40 psu)
flows in at the top of the L and out at the end of its foot. The C, O and O have
no boundaries, so they keep their starting ~35 psu. Only the L changes.

Interior segments need CrocoDash's `interior-obc-segments` branch (issue #338) and
the regional_mom6 fix on `interior-segment-corners`. The first cell puts both
worktrees ahead of the installed packages, so run it with the `dev_cd` kernel.
Forcings use the synthetic `reference_ocean` product, which takes seconds and needs
no downloads.

In [ ]:
import sys
from pathlib import Path

WT = Path("/glade/u/home/manishrv/documents/croc/regional_mom_workflows")
CROCODASH = WT / "CrocoDash-worktrees/interior-obc-v2"
RM6 = WT / "regional-mom6-worktrees/interior-segment-corners"
sys.path[:0] = [str(CROCODASH), str(RM6)]

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import regional_mom6
from regional_mom6.segment import Segment

import CrocoDash
from CrocoDash.case import Case
from CrocoDash.grid import Grid
from CrocoDash.topo import Topo
from CrocoDash.vgrid import VGrid

assert Path(CrocoDash.__file__).is_relative_to(CROCODASH), CrocoDash.__file__
assert Path(regional_mom6.__file__).is_relative_to(RM6), regional_mom6.__file__

HERE = Path("/glade/u/home/manishrv/documents/croc/dev/cool_demo")
CESMROOT = Path("~/work/installs/CESM_crocodash").expanduser()
CASEROOT = Path("~/croc_cases/cool_interior_obc").expanduser()
INPUTDIR = Path("~/scratch/croc_input/cool_interior_obc").expanduser()

## Grid and topography

The letters are one 0.1° cell wide. Refine the grid 4x so each stroke is 4 cells
wide: a one-cell channel is too narrow for MOM6 to carry a flow through it.

In [ ]:
R = 4  # each 0.1 deg cell becomes R x R cells
coarse = xr.open_dataset(HERE / "cool_topo.nc")
grid = Grid(
    resolution=0.1 / R,
    xstart=float(coarse.x.min()) - 0.05,
    lenx=0.1 * coarse.sizes["nx"],
    ystart=float(coarse.y.min()) - 0.05,
    leny=0.1 * coarse.sizes["ny"],
    name="cool",
)
topo = Topo(grid, min_depth=10.0, git=False)
topo.depth = np.kron(np.nan_to_num(coarse.depth.values), np.ones((R, R)))
vgrid = VGrid.uniform(nk=10, depth=20.0, name="cool")
print(grid.ny, "x", grid.nx, "cells,", int(topo.tmask.sum()), "ocean")

## Case

In [ ]:
case = Case(
    cesmroot=CESMROOT,
    caseroot=CASEROOT,
    inputdir=INPUTDIR,
    compset="1850_DATM%NYF_SLND_SICE_MOM6%REGIONAL_SROF_SGLC_SWAV",
    ocn_grid=grid,
    ocn_topo=topo,
    ocn_vgrid=vgrid,
    atm_grid_name="T62",
    machine="derecho",
    project="ncgd0011",
    ntasks_ocn=1,
)

## Two interior open boundaries on the L

On the coarse grid the L's stem is column 20 (rows 6-21) and its foot is row 6
(columns 20-25).

`Segment.from_lonlat` puts a segment on the T-row (or T-column) nearest
`fixed_lat` (or `fixed_lon`) and opens every T-cell from one end of the range to
the other. `ocean_side` says which side of the line is ocean; MOM6 treats the other
side as land. `topo=` checks that land caps both ends of each segment.

In [ ]:
hgrid = xr.open_dataset(case.supergrid_path)
tlon, tlat = grid.tlon.values, grid.tlat.values

# Fine T-cells of the L: its stem's top row and columns, its foot's east column and rows.
top, stem_w, stem_e = R * 22 - 1, R * 20, R * 21 - 1
foot_e, foot_s, foot_n = R * 26 - 1, R * 6, R * 7 - 1

inflow = Segment.from_lonlat(
    hgrid,
    axis="nyp",  # an east-west line across the top of the stem
    fixed_lat=tlat[top, stem_w],
    lon_range=(tlon[top, stem_w], tlon[top, stem_e]),
    ocean_side="south",
    segment_name="l_top",
    topo=topo,
)
outflow = Segment.from_lonlat(
    hgrid,
    axis="nxp",  # a north-south line across the end of the foot
    fixed_lon=tlon[foot_s, foot_e],
    lat_range=(tlat[foot_s, foot_e], tlat[foot_n, foot_e]),
    ocean_side="west",
    segment_name="l_foot",
    topo=topo,
)
for s in (inflow, outflow):
    print(s.segment_name, s.mom6_obc_position_string())

## Forcings

In [ ]:
case.configure_forcings(
    date_range=["2020-01-01 00:00:00", "2020-01-16 00:00:00"],
    boundaries=[inflow, outflow],
    product_name="reference_ocean",
    function_name="get_reference_ocean_data",
)
case.process_forcings()

## Salty water at the boundary

`reference_ocean` has about 35 psu everywhere. Set the water at the two segments:
- **l_top:** 40 psu flowing south into the L at 0.3 m/s.
- **l_foot:** carries the same volume out of the L.

Only the wet points change. Land points and every other variable stay as CrocoDash
wrote them, and CrocoDash's files are kept in `ocn/crocodash_original/`.

In [ ]:
import shutil

ocn = INPUTDIR / "ocn"
SALT_IN = 40.0  # psu
V_IN = 0.3  # m/s

# Open face widths from the supergrid: a segment opening T-cells lo..hi lies
# between supergrid corners 2*lo and 2*(hi+1), on the face row just past `top`
# (or the face column just past `foot_e`).
top_width = float(hgrid.dx.isel(nyp=2 * (top + 1), nx=slice(2 * stem_w, 2 * (stem_e + 1))).sum())
foot_height = float(hgrid.dy.isel(nxp=2 * (foot_e + 1), ny=slice(2 * foot_s, 2 * (foot_n + 1))).sum())
U_OUT = V_IN * top_width / foot_height
print(f"in {V_IN} m/s over {top_width:.0f} m, out {U_OUT:.3f} m/s over {foot_height:.0f} m")


def set_segment(n, **values):
    path = ocn / f"forcing_obc_segment_{n:03d}.nc"
    original = ocn / "crocodash_original" / path.name
    original.parent.mkdir(exist_ok=True)
    if not original.exists():
        shutil.copy2(path, original)
    ds = xr.open_dataset(original).load()
    for var, value in values.items():
        da = ds[f"{var}_segment_{n:03d}"]
        data = da.values.copy()
        data[data > -1e19] = value  # land points keep regional_mom6's -1e20 fill
        da.values = data
    ds.to_netcdf(path, unlimited_dims="time")


set_segment(1, salt=SALT_IN, u=0.0, v=-V_IN, eta=0.0)
set_segment(2, u=U_OUT, v=0.0, eta=0.0)

## Build and run on Derecho

`ntasks_ocn=1` sets only the ocean. Put every component on one task so the job
goes to the `develop` queue and starts right away. From a Derecho login node:

```bash
cd ~/croc_cases/cool_interior_obc
./xmlchange NTASKS=1,ROOTPE=0
./case.setup --reset
./case.build
./case.submit
```

The 15-day run writes daily surface fields to `cool_interior_obc.mom6.h.sfc.2020-01.nc`.

## Results

In [ ]:
scratch = Path("/glade/derecho/scratch") / Path.home().name
sfc_files = sorted(scratch.glob(f"archive/{CASEROOT.name}/ocn/hist/*.mom6.h.sfc.*.nc")) or sorted(
    scratch.glob(f"{CASEROOT.name}/run/*.mom6.h.sfc.*.nc")
)
sfc = xr.open_mfdataset(sfc_files)
sos = sfc.sos.where(topo.tmask.values > 0).load()
print(sos.sizes["time"], "daily means")

ramp = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list("blue", ramp)
cmap.set_bad("#e4e3df")  # land
ink = "#52514e"

days = [1, 5, 10, 15]
fig, axes = plt.subplots(1, len(days), figsize=(13, 3.8), sharey=True, layout="constrained")
for ax, day in zip(axes, days):
    m = ax.pcolormesh(sfc.xh, sfc.yh, sos.isel(time=day - 1), cmap=cmap, vmin=35, vmax=SALT_IN)
    ax.contour(sfc.xh, sfc.yh, topo.tmask.values, levels=[0.5], colors=ink, linewidths=0.6)
    ax.set_title(f"Day {day}", color=ink, fontsize=11)
    ax.set_xlim(177.1, 179.8)
    ax.set_ylim(55.5, 57.4)
    ax.tick_params(colors=ink, labelsize=8)
    for spine in ax.spines.values():
        spine.set_visible(False)
cb = fig.colorbar(m, ax=axes, shrink=0.8, pad=0.01)
cb.set_label("Sea surface salinity (psu)", color=ink)
cb.outline.set_visible(False)
cb.ax.tick_params(colors=ink, labelsize=8)
fig.suptitle("Salty water enters the L through its interior open boundaries", color=ink)
plt.show()

In [ ]:
from scipy import ndimage

# The letters are the four connected ocean regions; left to right they are C, O, O, L.
labels, n = ndimage.label(topo.tmask.values)
centers = ndimage.center_of_mass(topo.tmask.values, labels, range(1, n + 1))
letters = [labels == i + 1 for i in np.argsort([c[1] for c in centers])]
weight = np.cos(np.deg2rad(sfc.yh)) * xr.ones_like(sfc.xh)  # cell area on a lat-lon grid


def letter_mean(mask):
    w = weight.where(mask)
    return (sos * w).sum(["xh", "yh"]) / w.sum()


day = np.arange(1, sos.sizes["time"] + 1)
fig, ax = plt.subplots(figsize=(7, 3.5), layout="constrained")
for i, mask in enumerate(letters):
    is_l = i == len(letters) - 1
    ax.plot(
        day,
        letter_mean(mask),
        color="#2a78d6" if is_l else "#a3a29d",
        lw=2,
        label="L (two open boundaries)" if is_l else ("C, O, O (closed)" if i == 0 else None),
    )
ax.set_xlabel("Day", color=ink)
ax.set_ylabel("Mean surface salinity (psu)", color=ink)
ax.tick_params(colors=ink)
ax.grid(axis="y", color="#e4e3df", lw=0.8)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
ax.legend(frameon=False, labelcolor=ink)
plt.show()

## Animation

Saves `cool_salinity.gif` next to this notebook.

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter

l_mean = letter_mean(letters[-1]).values
coo_mean = letter_mean(letters[0] | letters[1] | letters[2]).values

fig = plt.figure(figsize=(8, 5.8), layout="constrained")
ax_map, ax_ts = fig.subplots(2, 1, height_ratios=[2.6, 1])
mesh = ax_map.pcolormesh(sfc.xh, sfc.yh, sos.isel(time=0), cmap=cmap, vmin=35, vmax=SALT_IN)
ax_map.contour(sfc.xh, sfc.yh, topo.tmask.values, levels=[0.5], colors=ink, linewidths=0.6)
ax_map.set_xlim(177.1, 179.8)
ax_map.set_ylim(55.5, 57.4)
ax_map.set_axis_off()
title = ax_map.set_title("", color=ink, fontsize=12)

# The two open boundaries, on the faces where MOM6 puts them.
x, y = hgrid.x.values, hgrid.y.values
face_top = y[2 * (top + 1), 0]
face_east = x[0, 2 * (foot_e + 1)]
ax_map.plot([x[0, 2 * stem_w], x[0, 2 * (stem_e + 1)]], [face_top] * 2, color="#eb6834", lw=4, solid_capstyle="butt")
ax_map.plot([face_east] * 2, [y[2 * foot_s, 0], y[2 * (foot_n + 1), 0]], color="#eb6834", lw=4, solid_capstyle="butt")
ax_map.annotate("in", (x[0, 2 * stem_w], face_top), xytext=(-4, 4), textcoords="offset points", ha="right", color=ink)
ax_map.annotate("out", (face_east, y[2 * foot_s, 0]), xytext=(6, 0), textcoords="offset points", va="bottom", color=ink)
cb = fig.colorbar(mesh, ax=ax_map, shrink=0.85, pad=0.01)
cb.set_label("Sea surface salinity (psu)", color=ink)
cb.outline.set_visible(False)
cb.ax.tick_params(colors=ink, labelsize=8)

ax_ts.plot(day, coo_mean, color="#a3a29d", lw=2, label="C, O, O (closed)")
ax_ts.plot(day, l_mean, color="#2a78d6", lw=2, label="L (two open boundaries)")
(marker,) = ax_ts.plot([], [], "o", color="#2a78d6", ms=8, mec="#fcfcfb", mew=2)
now = ax_ts.axvline(day[0], color=ink, lw=0.8, alpha=0.4)
ax_ts.set_xlabel("Day", color=ink)
ax_ts.set_ylabel("Mean (psu)", color=ink)
ax_ts.tick_params(colors=ink, labelsize=8)
ax_ts.grid(axis="y", color="#e4e3df", lw=0.8)
for spine in ("top", "right"):
    ax_ts.spines[spine].set_visible(False)
ax_ts.legend(frameon=False, labelcolor=ink, fontsize=9, loc="upper left")


def draw(i):
    mesh.set_array(np.ma.masked_invalid(sos.isel(time=i).values))
    title.set_text(f"Salty water flows through the L's interior open boundaries: day {day[i]}")
    marker.set_data([day[i]], [l_mean[i]])
    now.set_xdata([day[i], day[i]])
    return mesh, title, marker, now


frames = list(range(len(day))) + [len(day) - 1] * 4  # hold the last day
FuncAnimation(fig, draw, frames=frames).save(HERE / "cool_salinity.gif", writer=PillowWriter(fps=3))
plt.close(fig)